In [1]:
## Load Candidate Features
from pathlib import Path
import pandas as pd
import duckdb

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

feature_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "candidate_transaction_features_v2_cutoff_20170228.parquet"
)

con = duckdb.connect()

model_data = con.execute(
    "SELECT * FROM read_parquet(?)",
    [str(feature_path)],
).df()

print("Dataset shape:", model_data.shape)
print(f"Churn rate: {model_data['is_churn'].mean():.2%}")

Dataset shape: (970960, 9)
Churn rate: 8.99%


In [2]:
## Create a Stratified Random Holdout
from sklearn.model_selection import train_test_split

feature_columns = [
    "transaction_rows_90d",
    "transaction_days_90d",
    "cancel_rows_90d",
    "auto_renew_share_90d",
    "avg_plan_days_90d",
    "days_since_last_transaction_90d",
    "has_transaction_90d",
]

X = model_data[feature_columns]
y = model_data["is_churn"].astype("int8")

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print(f"Training churn rate: {y_train.mean():.2%}")
print(f"Validation churn rate: {y_valid.mean():.2%}")

Training rows: 776768
Validation rows: 194192
Training churn rate: 8.99%
Validation churn rate: 8.99%


In [3]:
## Train and Evaluate Logistic Regression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median", add_indicator=True),
    ),
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(max_iter=1000, random_state=42),
    ),
])

baseline_model.fit(X_train, y_train)

print("Baseline model fitted.")

Baseline model fitted.


In [4]:
import numpy as np
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    average_precision_score,
)

valid_probabilities = baseline_model.predict_proba(X_valid)[:, 1]

constant_probabilities = np.full(
    len(y_valid),
    y_train.mean(),
)

def evaluate_probabilities(name, y_true, probabilities):
    return {
        "model": name,
        "log_loss": log_loss(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "average_precision": average_precision_score(
            y_true, probabilities
        ),
    }

baseline_results = pd.DataFrame([
    evaluate_probabilities(
        "Constant probability",
        y_valid,
        constant_probabilities,
    ),
    evaluate_probabilities(
        "Logistic regression",
        y_valid,
        valid_probabilities,
    ),
])

baseline_results

,model,log_loss,roc_auc,average_precision
0,Constant probability,0.302403,0.500000,0.089942
1,Logistic regression,0.200084,0.854197,0.555505


In [5]:
assert np.isfinite(valid_probabilities).all()
assert (
    (valid_probabilities >= 0)
    & (valid_probabilities <= 1)
).all()

pd.Series(
    valid_probabilities,
    name="predicted_churn_probability",
).describe(
    percentiles=[0.01, 0.10, 0.50, 0.90, 0.99]
)

count    194192.000000
mean          0.089951
std           0.167631
min           0.000525
1%            0.015465
10%           0.015465
50%           0.029719
90%           0.225972
99%           0.837961
max           0.989577
Name: predicted_churn_probability, dtype: float64

In [6]:
## Compare Against an Availability-Only Model
availability_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

availability_model.fit(
    X_train[["has_transaction_90d"]],
    y_train,
)

availability_probabilities = availability_model.predict_proba(
    X_valid[["has_transaction_90d"]]
)[:, 1]

availability_result = pd.DataFrame([
    evaluate_probabilities(
        "Transaction availability only",
        y_valid,
        availability_probabilities,
    ),
])

comparison_results = pd.concat(
    [baseline_results, availability_result],
    ignore_index=True,
).sort_values("log_loss").reset_index(drop=True)

comparison_results

,model,log_loss,roc_auc,average_precision
0,Logistic regression,0.200084,0.854197,0.555505
1,Transaction availability only,0.254391,0.629293,0.286939
2,Constant probability,0.302403,0.500000,0.089942


In [7]:
import json

report_dir = PROJECT_ROOT / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

comparison_results.to_csv(
    report_dir / "baseline_model_comparison.csv",
    index=False,
)

evaluation_config = {
    "validation_method": "stratified random holdout",
    "validation_fraction": 0.2,
    "random_state": 42,
    "training_rows": len(X_train),
    "validation_rows": len(X_valid),
    "cohort_source": "train_v2.csv",
    "transaction_source": "transactions.csv",
    "feature_cutoff": "2017-02-28",
    "features": feature_columns,
    "temporal_validation_completed": False,
    "independent_test_evaluation_completed": False,
}

(report_dir / "baseline_evaluation_config.json").write_text(
    json.dumps(evaluation_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Model comparison and evaluation configuration saved.")

Model comparison and evaluation configuration saved.
